# 05 - Revenue Scenario Modelling

This notebook models forward-looking revenue scenarios:
- Create price and generation scenarios
- Run Monte Carlo simulations for revenue uncertainty
- Calculate revenue distributions (P10/P50/P90)
- Compare scenario outcomes
- Risk analysis (VaR, CVaR)

## Outputs
- Revenue scenarios and distributions
- Risk metrics
- Scenario comparison visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.append('../')

from src.models import MonteCarloSimulator, monte_carlo_revenue_analysis
from src.scenarios import create_generation_scenarios, create_price_scenarios, create_combined_scenarios
from src.utils import load_config, plot_revenue_distribution, plot_scenario_comparison

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
config = load_config('../configs/modelling_config.yaml')
data_path = Path(config.get('data.processed_path', '../data_processed'))
df = pd.read_csv(data_path / 'features.csv', index_col=0, parse_dates=True)

asset_capacity_mw = config.get('asset.capacity_mw', 100)
asset_type = config.get('asset.type', 'offshore_wind')

print(f"Data shape: {df.shape}")
print(f"Asset: {asset_capacity_mw}MW {asset_type}")

## 2. Create Price Scenarios

In [ ]:
# Create price scenarios
price_scenarios = create_price_scenarios(df, base_scenario='historical')

print(f"Created {len(price_scenarios)} price scenarios:")
for name, prices in price_scenarios.items():
    print(f"  {name}: mean = £{prices.mean():.2f}/MWh")

In [ ]:
# Plot price scenarios
fig, ax = plt.subplots(figsize=(12, 6))
for name, prices in price_scenarios.items():
    monthly_avg = prices.resample('M').mean()
    ax.plot(monthly_avg.index, monthly_avg.values, label=name, alpha=0.7, linewidth=2)
ax.set_ylabel('Price (£/MWh)')
ax.set_title('Monthly Average Price Scenarios')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Create Generation Scenarios

In [ ]:
# Create generation scenarios
gen_scenarios = create_generation_scenarios(df, asset_type=asset_type, asset_capacity_mw=asset_capacity_mw)

print(f"\nCreated {len(gen_scenarios)} generation scenarios:")
for name, gen in gen_scenarios.items():
    cf = gen.sum() / (asset_capacity_mw * len(gen))
    print(f"  {name}: CF = {cf:.2%}, Total = {gen.sum():,.0f} MWh")

In [ ]:
# Plot generation scenarios
fig, ax = plt.subplots(figsize=(12, 6))
for name, gen in gen_scenarios.items():
    monthly_avg = gen.resample('M').mean()
    ax.plot(monthly_avg.index, monthly_avg.values, label=name, alpha=0.7, linewidth=2)
ax.set_ylabel('Generation (MWh)')
ax.set_title('Monthly Average Generation Scenarios')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Monte Carlo Revenue Simulation

In [ ]:
# Run Monte Carlo simulation
n_simulations = config.get('monte_carlo.n_simulations', 1000)
price_volatility = config.get('monte_carlo.price_volatility', 0.2)
generation_uncertainty = config.get('monte_carlo.generation_uncertainty', 0.1)

mc_results = monte_carlo_revenue_analysis(
    df,
    asset_type=asset_type,
    n_simulations=n_simulations,
    price_volatility=price_volatility,
    generation_uncertainty=generation_uncertainty
)

In [ ]:
# Display results
base_case = mc_results['base_case']
print("\n" + "="*60)
print("MONTE CARLO REVENUE DISTRIBUTION")
print("="*60)
print(f"Mean Revenue:     £{base_case['mean']:,.0f}")
print(f"Std Dev:          £{base_case['std']:,.0f}")
print(f"P10 (pessimistic): £{base_case['p10']:,.0f}")
print(f"P50 (median):      £{base_case['p50']:,.0f}")
print(f"P90 (optimistic):  £{base_case['p90']:,.0f}")
print(f"Min:              £{base_case['min']:,.0f}")
print(f"Max:              £{base_case['max']:,.0f}")

# Risk metrics
risk_metrics = mc_results['risk_metrics']
print("\n" + "="*60)
print("RISK METRICS")
print("="*60)
print(f"VaR 95%:  £{risk_metrics['var_95']:,.0f}")
print(f"CVaR 95%: £{risk_metrics['cvar_95']:,.0f}")
print(f"VaR 99%:  £{risk_metrics['var_99']:,.0f}")
print(f"CVaR 99%: £{risk_metrics['cvar_99']:,.0f}")

In [ ]:
# Plot revenue distribution
plot_revenue_distribution(
    base_case['revenues'],
    title=f'Revenue Distribution ({n_simulations} simulations)',
    percentiles=[10, 50, 90]
)

## 5. Combined Scenario Analysis

In [ ]:
# Create combined scenarios
combined_scenarios = create_combined_scenarios(df, asset_type=asset_type)

# Calculate revenue for each scenario
scenario_revenues = {}
for name, scenario_data in combined_scenarios.items():
    revenue = (scenario_data['price'] * scenario_data['generation']).sum()
    scenario_revenues[name] = {
        'total_revenue': revenue,
        'avg_price': scenario_data['price'].mean(),
        'total_generation': scenario_data['generation'].sum()
    }

scenario_df = pd.DataFrame(scenario_revenues).T
print("\nCombined Scenario Results:")
print(scenario_df)

In [ ]:
# Plot scenario comparison
plot_scenario_comparison(scenario_df, metric='total_revenue')

## 6. Sensitivity Analysis

In [ ]:
# Price sensitivity
price_multipliers = np.linspace(0.7, 1.3, 7)
sensitivity_results = []

base_price = df['price']
base_gen = df[asset_type]

for mult in price_multipliers:
    revenue = (base_price * mult * base_gen).sum()
    sensitivity_results.append({
        'price_multiplier': mult,
        'revenue': revenue,
        'price_change_pct': (mult - 1) * 100
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sensitivity_df['price_change_pct'], sensitivity_df['revenue'], marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Price Change (%)')
ax.set_ylabel('Annual Revenue (£)')
ax.set_title('Revenue Sensitivity to Price Changes')
ax.grid(True, alpha=0.3)
ax.axvline(x=0, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
# Save scenario results
output_path = Path(config.get('outputs.reports_path', '../reports'))
output_path.mkdir(parents=True, exist_ok=True)

# Save Monte Carlo results
mc_summary = pd.DataFrame({
    'metric': ['mean', 'std', 'p10', 'p50', 'p90', 'min', 'max'],
    'value': [base_case[m] for m in ['mean', 'std', 'p10', 'p50', 'p90', 'min', 'max']]
})
mc_summary.to_csv(output_path / 'monte_carlo_summary.csv', index=False)

# Save scenario comparison
scenario_df.to_csv(output_path / 'combined_scenarios.csv')

print(f"\n✓ Results saved to {output_path}")

## Summary

Scenario modelling complete!

**Key outputs:**
- Revenue distribution from Monte Carlo simulation
- P10/P50/P90 revenue estimates
- Risk metrics (VaR, CVaR)
- Multiple scenario comparisons
- Sensitivity analysis

**Insights:**
- Revenue uncertainty quantified
- Downside risk measured
- Scenario impacts compared

This completes the revenue modelling workflow!